In [21]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers,Sequential

In [22]:
train_ds=tf.keras.utils.image_dataset_from_directory(
    r"C:\Users\raipa\Downloads\cats_dogs_light\train",
    seed=42,
    image_size=(160,160),
    batch_size=16,
    label_mode="binary"
)
test_ds=tf.keras.utils.image_dataset_from_directory(
    r"C:\Users\raipa\Downloads\cats_dogs_light\test",
    seed=42,
    image_size=(160,160),
    batch_size=16,
    label_mode="binary"
)

print("Classes:",train_ds.class_names)

Found 1000 files belonging to 2 classes.
Found 400 files belonging to 2 classes.
Classes: ['cats', 'dogs']


In [38]:
data_augmentation=Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1)
])
preprocess=tf.keras.applications.mobilenet_v2.preprocess_input

base_model=tf.keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(160,160,3)
)
base_model.trainable=False

In [25]:
inputs = layers.Input(shape=(160, 160, 3))
x = data_augmentation(inputs)
x = preprocess(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)
model = tf.keras.Model(inputs, outputs)
model.summary()

Model: "model_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_11 (InputLayer)       [(None, 160, 160, 3)]     0         
                                                                 
 sequential_2 (Sequential)   (None, 160, 160, 3)       0         
                                                                 
 tf.math.truediv_7 (TFOpLamb  (None, 160, 160, 3)      0         
 da)                                                             
                                                                 
 tf.math.subtract_7 (TFOpLam  (None, 160, 160, 3)      0         
 bda)                                                            
                                                                 
 mobilenetv2_1.00_224 (Funct  (None, None, None, 1280)  2257984  
 ional)                                                          
                                                           

In [26]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"])
early_stopping=tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True)
history=model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=10,
    callbacks=[early_stopping],
    verbose=1)

loss,accuracy=model.evaluate(test_ds,verbose=1)
print("Validation Loss :",loss)
print("validation Accuracy:",accuracy)

Epoch 1/10
63/63 [==============================] - 20s 212ms/step - loss: 0.2947 - accuracy: 0.8740 - val_loss: 0.1629 - val_accuracy: 0.9600
Epoch 2/10
63/63 [==============================] - 12s 194ms/step - loss: 0.1423 - accuracy: 0.9490 - val_loss: 0.1322 - val_accuracy: 0.9575
Epoch 3/10
63/63 [==============================] - 12s 194ms/step - loss: 0.1168 - accuracy: 0.9490 - val_loss: 0.1238 - val_accuracy: 0.9625
Epoch 4/10
63/63 [==============================] - 12s 193ms/step - loss: 0.1014 - accuracy: 0.9650 - val_loss: 0.1197 - val_accuracy: 0.9650
Epoch 5/10
63/63 [==============================] - 12s 192ms/step - loss: 0.0833 - accuracy: 0.9750 - val_loss: 0.1199 - val_accuracy: 0.9650
Epoch 6/10
63/63 [==============================] - 12s 195ms/step - loss: 0.0796 - accuracy: 0.9680 - val_loss: 0.1086 - val_accuracy: 0.9650
Epoch 7/10
63/63 [==============================] - 12s 196ms/step - loss: 0.0653 - accuracy: 0.9790 - val_loss: 0.1084 - val_accuracy: 0.9625

In [35]:
img=tf.keras.utils.load_img(r"C:\Users\raipa\OneDrive\Desktop\New folder (2)\dog.jpg",target_size=(160,160))
img_to_array=tf.keras.utils.img_to_array(img)
img_exp=np.expand_dims(img_to_array,axis=0)
prob=model.predict(img_exp)[0][0]
print(prob)
if prob >=0.5 :
    print("Dog")
else :
    print("Cat")

1/1 [==============================] - 0s 39ms/step
0.9988287
Dog
